# Clip-on card tray

This is the single source notebook for the tray: dimensions, geometry, interactive view, CAD exports, and static previews.

The current tray is 240 × 200 mm, with a 2 mm floor, 4 mm drop, one card bay, and one accessory bay. The clips are 100 mm wide, with 4 mm upper tabs, 12.4 mm lower tabs, and 10 mm webs and skirt posts. The skirt posts reach the bottom of the 83 mm skirt and have nominally zero gap for an adjustable shim fit.

Run the cells from top to bottom after changing parameters. The exported files are generated outputs; load capacity has not been established.


## 1. Dimensions

The tabletop is 24 mm thick, its edge overhangs the skirt by 35 mm, and the skirt drops 83 mm below the tabletop underside. Your 2 mm floor and 4 mm tray drop are preserved. With the current 12 mm rim, its top is 8 mm above the tabletop.


In [1]:
import cadquery as cq
import math
from IPython.display import display

tray_width = 240.0
tray_depth = 200.0
floor_thickness = 2.0
tray_drop = 4.0             # floor top is this far below table top
wall_thickness = 3.0
wall_height = 12.0          # height above the tray floor

table_thickness = 24.0
table_overhang = 35.0       # from table edge inward to skirt face
clip_clearance = 0.0
clip_thickness = 10.0
lower_tab_thickness = 12.4
clip_width = 100.0
skirt_gap = 0.0
skirt_drop = 83.0

organizer_height = 2.0
organizer_thickness = 3.0
brace_width = 12.0
brace_thickness = 8.0
cross_brace_size = 8.0
post_depth = 10.0

# Upper tabs are independent of the thin tray floor.
top_tab_thickness = 4.0
clip_outer_margin = 0.0


## 2. Tray floor and rim

`box` makes a centred rectangular solid. `translate` places it in the model. `union` joins touching or overlapping solids into one part.

In [2]:
def block(length_x, length_y, height, x, y, z):
    return (cq.Workplane("XY")
            .box(length_x, length_y, height)
            .translate((x, y, z)))

floor_top = -tray_drop
floor_bottom = floor_top - floor_thickness
floor = block(tray_width, tray_depth, floor_thickness,
              0, tray_depth / 2, (floor_top + floor_bottom) / 2)
wall_z = floor_top + wall_height / 2
side_x = tray_width / 2 - wall_thickness / 2

tray = floor
tray = tray.union(block(wall_thickness, tray_depth, wall_height,
                        -side_x, tray_depth / 2, wall_z))
tray = tray.union(block(wall_thickness, tray_depth, wall_height,
                        side_x, tray_depth / 2, wall_z))
tray = tray.union(block(tray_width, wall_thickness, wall_height,
                        0, wall_thickness / 2, wall_z))
tray = tray.union(block(tray_width, wall_thickness, wall_height,
                        0, tray_depth - wall_thickness / 2, wall_z))

display(tray)

## 3. Clips and full-height skirt posts

Each clip catches the tabletop lip. A vertical post beside the skirt reaches from the tabletop underside to the skirt bottom, 107 mm below the table's upper surface.


In [3]:
clip_centres_x = tray_width / 2 - clip_width / 2 - clip_outer_margin
table_bottom_z = -table_thickness
lower_top = table_bottom_z - clip_clearance
lower_bottom = lower_top - lower_tab_thickness
web_bottom = min(lower_bottom, floor_bottom) - 1.0
web_top = max(floor_top + wall_height, top_tab_thickness)
web_height = web_top - web_bottom
web_z = (web_top + web_bottom) / 2

skirt_face_y = -table_overhang
support_y = skirt_face_y + skirt_gap + clip_thickness / 2
skirt_bottom_z = table_bottom_z - skirt_drop
post_z = (table_bottom_z + skirt_bottom_z) / 2

for x in (-clip_centres_x, clip_centres_x):
    top_tab = block(clip_width, table_overhang, top_tab_thickness,
                    x, -table_overhang / 2, top_tab_thickness / 2)
    lower_tab = block(clip_width, table_overhang, lower_tab_thickness,
                      x, -table_overhang / 2,
                      (lower_top + lower_bottom) / 2)
    edge_web = block(clip_width, clip_thickness, web_height,
                     x, clip_thickness / 2, web_z)
    skirt_post = block(clip_width, clip_thickness, skirt_drop,
                       x, support_y, post_z)
    tray = tray.union(top_tab).union(lower_tab).union(edge_web).union(skirt_post)


## 4. Diagonal braces and X cross-bracing

Two side ribs run from the skirt posts to the outer underside of the tray. Two additional members connect opposite corners of this inclined frame, forming an X. They join where they cross.


In [4]:
brace_start_y = support_y
brace_start_z = skirt_bottom_z
brace_end_y = tray_depth - wall_thickness / 2
brace_end_z = floor_bottom

dy = brace_end_y - brace_start_y
dz = brace_end_z - brace_start_z
brace_length = math.hypot(dy, dz)
normal_y = -dz / brace_length
normal_z = dy / brace_length
half_thickness = brace_thickness / 2

brace_outline = [
    (brace_start_y + normal_y * half_thickness,
     brace_start_z + normal_z * half_thickness),
    (brace_end_y + normal_y * half_thickness,
     brace_end_z + normal_z * half_thickness),
    (brace_end_y - normal_y * half_thickness,
     brace_end_z - normal_z * half_thickness),
    (brace_start_y - normal_y * half_thickness,
     brace_start_z - normal_z * half_thickness),
]

for x in (-clip_centres_x, clip_centres_x):
    diagonal = (cq.Workplane("YZ")
                .polyline(brace_outline).close()
                .extrude(brace_width)
                .translate((x - brace_width / 2, 0, 0)))
    tray = tray.union(diagonal)

def cross_beam(start, end, size):
    direction = tuple(b - a for a, b in zip(start, end))
    length = math.sqrt(sum(value * value for value in direction))
    plane = cq.Plane(origin=start, normal=direction)
    return cq.Workplane(plane).rect(size, size).extrude(length)

# Both members lie across the inclined frame between the two side braces.
# They overlap at the centre, forming a joined X.
for x in (-clip_centres_x, clip_centres_x):
    start = (x, brace_start_y, brace_start_z)
    end = (-x, brace_end_y, brace_end_z)
    tray = tray.union(cross_beam(start, end, cross_brace_size))


## 5. Middle support posts

At half the tray depth, a vertical post rises from each diagonal to the floor underside. Slight overlaps join the posts securely to both solids.


In [5]:
post_y = tray_depth / 2
brace_center_z = brace_start_z + (post_y - brace_start_y) * dz / dy
brace_top_z = brace_center_z + normal_z * half_thickness
post_bottom_z = brace_top_z - 2.0
post_top_z = floor_bottom + 1.0
post_height = post_top_z - post_bottom_z
post_z = (post_bottom_z + post_top_z) / 2

for x in (-clip_centres_x, clip_centres_x):
    post = block(brace_width, post_depth, post_height,
                 x, post_y, post_z)
    tray = tray.union(post)


## 6. Card and accessory bays

One 78 × 104 mm outside card bay (72 × 98 mm inside) fits a single standard-sized card deck. One larger accessory bay replaces the three small bins; the remaining floor stays open. The card size assumes sleeved poker-size cards.

In [6]:
def rail_x(x0, x1, y):
    return block(x1 - x0, organizer_thickness, organizer_height,
                 (x0 + x1) / 2, y, floor_top + organizer_height / 2)

def rail_y(x, y0, y1):
    return block(organizer_thickness, y1 - y0, organizer_height,
                 x, (y0 + y1) / 2, floor_top + organizer_height / 2)

walls = []
# One deck bay: 78 x 104 mm outside, about 72 x 98 mm clear inside.
card_x0, card_x1 = -110.0, -32.0
card_y0, card_y1 = 18.0, 122.0
walls += [rail_y(card_x0, card_y0, card_y1),
          rail_y(card_x1, card_y0, card_y1),
          rail_x(card_x0, card_x1, card_y0),
          rail_x(card_x0, card_x1, card_y1)]

# One shared accessory bay; the rest of the tray remains open.
accessory_x0, accessory_x1 = -32.0, 110.0
accessory_y0, accessory_y1 = 18.0, 88.0
walls += [rail_y(accessory_x0, accessory_y0, accessory_y1),
          rail_y(accessory_x1, accessory_y0, accessory_y1),
          rail_x(accessory_x0, accessory_x1, accessory_y0),
          rail_x(accessory_x0, accessory_x1, accessory_y1)]

for wall in walls:
    tray = tray.union(wall)

display(tray)

## 7. Export

STEP keeps editable CAD geometry; STL is a mesh for a slicer. Export only after checking the fit and organizer sizes.

In [7]:
cq.exporters.export(tray, "clip_on_tray.step")
cq.exporters.export(tray, "clip_on_tray.stl")

## 8. Static previews

Regenerate the existing overview and underside images from the same model.


In [8]:
from cadquery.vis import show

view_scale = max(tray_width, tray_depth)
view_focus = (0, tray_depth / 2 - 15, -30)
common_view = dict(
    title="Clip-on card tray", interact=False,
    focus=view_focus, viewup=(0, 0, 1),
    roll=0, elevation=0, azimuth=0,
    width=1400, height=900,
    bgcolor=(0.97, 0.98, 1.0), gradient=False, edges=False,
)
show(tray, screenshot="clip_on_tray_preview.png",
     position=(view_scale * 1.6, tray_depth * 2.1, view_scale * 2.4),
     **common_view)
show(tray, screenshot="clip_on_tray_braces.png",
     position=(view_scale * 1.6, tray_depth * 2.1, -view_scale * 2),
     **common_view)
